# CareVoice -- Offline Clinical Intake Assistant

Offline multilingual clinical intake assistant using Gemma 4 on-device.
Runs 10 clinical scenarios across English, Spanish, and French.

Kaggle setup:
  1. Add model: Models > Google > Gemma 4 > Transformers > gemma-4-e4b-it
     (kernel-metadata.json model_sources: google/gemma-4/transformers/gemma-4-e4b-it/1)
  2. Enable internet (needed to upgrade transformers for Gemma 4 support)
  3. GPU optional -- CPU fallback uses bfloat16 (~8GB RAM, fits Kaggle CPU tier)

## Install dependencies

In [ ]:
# Kaggle ships transformers 5.0.0 which lacks Gemma 4 in CONFIG_MAPPING.
# Upgrade to latest stable PyPI release (more reliable than git HEAD).
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers", "accelerate>=0.34.0",
])

import sys, os, json
from pathlib import Path

print("Python:", sys.version)
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    sm = cap[0] * 10 + cap[1]
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{sm})")

import importlib
import transformers as _tf_check
importlib.reload(_tf_check)
import transformers
print("transformers:", transformers.__version__)

# Confirm Gemma 4 is registered
from transformers import CONFIG_MAPPING
has_gemma4 = any("gemma4" in k.lower() or "gemma-4" in k.lower() for k in CONFIG_MAPPING)
print(f"Gemma 4 in CONFIG_MAPPING: {has_gemma4}")

## Locate Gemma 4 model

In [ ]:
# Kaggle mounts model_sources under /kaggle/input/models/{owner}/{model-slug}/
MODEL_PATH = None
for p in [
    "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1",
    "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1",
]:
    if Path(p).exists():
        print(f"Model found at: {p}")
        MODEL_PATH = p
        break

if MODEL_PATH is None:
    raise RuntimeError("Model not found. Add google/gemma-4 > transformers > gemma-4-e4b-it in notebook settings.")

## Load model and processor

In [ ]:
# Correct API per HuggingFace docs (transformers v5.5+):
#   AutoProcessor (not AutoTokenizer)
#   AutoModelForImageTextToText (not AutoModelForCausalLM)
#   bfloat16 -- 4B model = 8GB, fits in Kaggle CPU RAM (16GB)
# Ref: https://huggingface.co/docs/transformers/model_doc/gemma4

from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

print(f"Loading processor from {MODEL_PATH}...")
processor = AutoProcessor.from_pretrained(MODEL_PATH, padding_side="left")

# Determine device: use GPU only if sm_70+ (P100 is sm_60, incompatible with PyTorch 2.10+)
USE_GPU = False
if torch.cuda.is_available():
    sm = torch.cuda.get_device_capability(0)
    USE_GPU = (sm[0] >= 7)

device_map = "auto" if USE_GPU else "cpu"
print(f"Loading model in bfloat16 (device_map={device_map!r})...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    attn_implementation="eager",
)
model.eval()
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Model loaded. Device: {DEVICE}")
if USE_GPU:
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## CareVoice system prompt and generation helper

In [ ]:
SYSTEM_PROMPT = (
    "You are CareVoice, an offline clinical intake assistant running on a health "
    "worker's device in a resource-constrained setting.\n\n"
    "LANGUAGE RULE (CRITICAL): Detect the language of the patient's LAST message. "
    "Respond ONLY in that exact language. If the patient writes in English, respond "
    "in English. If Spanish, respond in Spanish. If French, respond in French. "
    "Never switch languages mid-conversation.\n\n"
    "Your role:\n"
    "- Collect structured patient intake info through a calm, empathetic conversation\n"
    "- Ask ONE clear question at a time\n"
    "- NEVER diagnose, prescribe, or give specific medical advice\n"
    "- Flag emergencies immediately (chest pain, breathing difficulty, stroke signs, "
    "suicidal ideation, severe bleeding, anaphylaxis)\n\n"
    "FIELD NAMES — use EXACTLY these strings for extracted_field:\n"
    "  chief_complaint       (primary reason for visit — use this on turn 1)\n"
    "  symptom_duration      (how long symptoms have lasted)\n"
    "  symptom_severity      (numeric pain/severity score)\n"
    "  medical_history       (past conditions)\n"
    "  current_medications   (drugs currently taken)\n"
    "  allergies             (known drug or other allergies)\n"
    "  associated_symptoms   (other symptoms beyond chief complaint)\n\n"
    "OUTPUT FORMAT — always respond with valid JSON only, no prose outside the JSON:\n"
    '{"message": "<next question or acknowledgement in PATIENT language>", '
    '"extracted_field": "<exact field name from list above, or null>", '
    '"extracted_value": "<extracted value as string, or null>", '
    '"confidence": <0.0-1.0>, '
    '"red_flag": <true/false>, '
    '"red_flag_reason": "<reason or null>", '
    '"intake_complete": <true/false>}'
)


def generate_response(conversation: list[dict], max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]
    for msg in conversation:
        messages.append({
            "role": msg["role"],
            "content": [{"type": "text", "text": msg["content"]}],
        })

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(DEVICE)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    return processor.decode(new_tokens, skip_special_tokens=True)


# Canonical field names — any synonym gets mapped to the canonical name
FIELD_ALIASES = {
    "chief_complaint_description": "chief_complaint",
    "presenting_complaint": "chief_complaint",
    "main_complaint": "chief_complaint",
    "primary_complaint": "chief_complaint",
    "symptoms": "chief_complaint",
    "symptom": "chief_complaint",
    "complaint": "chief_complaint",
    "pain_severity": "symptom_severity",
    "severity": "symptom_severity",
    "fever": "symptom_severity",
    "past_medical_history": "medical_history",
    "medications": "current_medications",
    "medication": "current_medications",
    "allergy": "allergies",
    "other_symptoms": "associated_symptoms",
    "associated_symptom": "associated_symptoms",
}

VALID_FIELDS = {
    "chief_complaint", "symptom_duration", "symptom_severity",
    "medical_history", "current_medications", "allergies", "associated_symptoms",
}


def normalise_field(raw: str | None) -> str | None:
    if not raw:
        return None
    raw = raw.strip().lower().replace(" ", "_")
    if raw in VALID_FIELDS:
        return raw
    return FIELD_ALIASES.get(raw)


def parse_json(raw: str) -> dict:
    import re
    raw = raw.strip()
    try:
        d = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        if m:
            try:
                d = json.loads(m.group())
            except json.JSONDecodeError:
                d = {}
        else:
            d = {}
    if not d:
        return {
            "message": raw, "extracted_field": None, "extracted_value": None,
            "confidence": 0.0, "red_flag": False, "red_flag_reason": None, "intake_complete": False,
        }
    # Normalise field name
    d["extracted_field"] = normalise_field(d.get("extracted_field"))
    return d

## Run all 10 scenarios

In [ ]:
SCENARIOS = [
    {"scenario_id": "scenario_01", "language": "en",
     "description": "Adult with acute chest discomfort",
     "turns": [
         "I've been having this tightness in my chest since this morning.",
         "It's about a 6 out of 10. It gets worse when I walk fast.",
         "I had a heart attack two years ago. I take aspirin and metoprolol daily.",
     ]},
    {"scenario_id": "scenario_02", "language": "es",
     "description": "Child with fever and cough (Spanish)",
     "turns": [
         "Mi hija tiene fiebre y tos desde hace tres dias.",
         "Tiene ocho anos. La fiebre es de 38.5 grados.",
         "No tiene alergias que yo sepa. No toma ningun medicamento.",
     ]},
    {"scenario_id": "scenario_03", "language": "fr",
     "description": "Pregnant woman with headache (French)",
     "turns": [
         "J'ai un mal de tete tres fort depuis hier soir. Je suis enceinte de 7 mois.",
         "C'est tres intense, environ 8 sur 10. J'ai aussi une vision floue.",
         "Je prends des vitamines prenatales. Pas d'allergies connues.",
     ]},
    {"scenario_id": "scenario_04", "language": "en",
     "description": "Adult with abdominal pain",
     "turns": [
         "I've had stomach pain and diarrhea for two days now.",
         "The pain is around a 4. It's crampy, mostly in my lower belly.",
         "I don't have any chronic conditions. No regular medications.",
     ]},
    {"scenario_id": "scenario_05", "language": "en",
     "description": "Elderly with sudden confusion and fall (stroke red flag)",
     "turns": [
         "My father is 78. He suddenly got confused this morning and fell.",
         "He can't seem to move his left arm properly and his speech is slurred.",
         "He has high blood pressure and takes lisinopril. No known allergies.",
     ]},
    {"scenario_id": "scenario_06", "language": "es",
     "description": "Adult with rash after shellfish (anaphylaxis risk)",
     "turns": [
         "Tengo una erupcion en los brazos y el pecho desde ayer. Me pica mucho.",
         "Empezo despues de que comi mariscos en la noche. Tengo hinchazon en los labios.",
         "Nunca me habia pasado esto. No tomo medicamentos.",
     ]},
    {"scenario_id": "scenario_07", "language": "en",
     "description": "Adult with mental health crisis",
     "turns": [
         "I haven't been sleeping and I feel like I can't go on anymore.",
         "I've been feeling this way for about three weeks. I've had these thoughts before.",
         "I'm not taking anything right now. I stopped my antidepressants a month ago.",
     ]},
    {"scenario_id": "scenario_08", "language": "fr",
     "description": "Diabetic with non-healing foot wound (French)",
     "turns": [
         "J'ai une plaie au pied droit qui ne guerit pas depuis deux semaines.",
         "Je suis diabetique de type 2 depuis dix ans. La blessure commence a sentir mauvais.",
         "Je prends de la metformine et de l'insuline. Je suis allergique a la penicilline.",
     ]},
    {"scenario_id": "scenario_09", "language": "en",
     "description": "Adult with respiratory infection",
     "turns": [
         "I have a sore throat, runny nose, and a low fever for the past four days.",
         "My temperature was 37.8 this morning. The throat pain is about a 3 out of 10.",
         "I'm generally healthy. I'm not on any medications. No drug allergies.",
     ]},
    {"scenario_id": "scenario_10", "language": "en",
     "description": "Elderly patient with medication side effects",
     "turns": [
         "I started a new blood pressure tablet last week and I've been feeling dizzy.",
         "The dizziness is worst when I stand up. I almost fell yesterday.",
         "I'm 72. I take amlodipine, atorvastatin, and now this new one -- ramipril.",
     ]},
]


def run_scenario(scenario: dict) -> dict:
    sid = scenario["scenario_id"]
    print(f"\n{'='*60}")
    print(f"[{sid}] {scenario['description']}")
    print('='*60)

    conversation = []
    records = {
        "chief_complaint": "", "symptom_duration": "", "symptom_severity": 0,
        "medical_history": [], "current_medications": [], "allergies": [],
        "associated_symptoms": [], "red_flags": [],
    }
    turn_results = []

    for i, patient_input in enumerate(scenario["turns"]):
        print(f"\n  Patient: {patient_input}")
        conversation.append({"role": "user", "content": patient_input})

        raw = generate_response(conversation)
        parsed = parse_json(raw)

        assistant_msg = parsed.get("message", "")
        print(f"  CareVoice: {assistant_msg}")
        if parsed.get("red_flag"):
            print(f"  [!] RED FLAG: {parsed.get('red_flag_reason')}")

        field_name = parsed.get("extracted_field", "")
        field_val = parsed.get("extracted_value")
        if field_name and field_val is not None:
            if field_name in ("medical_history", "current_medications", "allergies", "associated_symptoms"):
                vals = field_val if isinstance(field_val, list) else [str(field_val)]
                records[field_name].extend(vals)
            elif field_name in records:
                records[field_name] = field_val

        if parsed.get("red_flag") and parsed.get("red_flag_reason"):
            if parsed["red_flag_reason"] not in records["red_flags"]:
                records["red_flags"].append(parsed["red_flag_reason"])

        # For crisis scenarios: if model raises a red_flag but defers extraction,
        # treat the patient's first message as the chief complaint fallback.
        if i == 0 and not records["chief_complaint"] and parsed.get("red_flag"):
            records["chief_complaint"] = patient_input[:120]

        conversation.append({"role": "assistant", "content": assistant_msg})
        turn_results.append({
            "turn": i, "patient_input": patient_input, "assistant_message": assistant_msg,
            "extracted_field": field_name, "confidence": parsed.get("confidence", 0.0),
            "red_flag": parsed.get("red_flag", False),
        })

    print("\n  --- Provider Summary ---")
    for k in ("chief_complaint", "symptom_duration", "symptom_severity",
              "medical_history", "current_medications", "allergies", "red_flags"):
        v = records[k]
        if v:
            print(f"  {k.upper()}: {v}")

    # Pass: chief complaint captured (directly or via crisis fallback) + all turns completed
    passed = bool(records["chief_complaint"]) and len(turn_results) == len(scenario["turns"])
    print(f"\n  PASS: {passed}")

    avg_conf = sum(t["confidence"] for t in turn_results) / len(turn_results) if turn_results else 0
    return {
        "scenario_id": sid, "description": scenario["description"],
        "passed_d4_criterion": passed, "final_record": records,
        "turns": turn_results, "avg_confidence": round(avg_conf, 3),
    }


all_results = []
for scenario in SCENARIOS:
    result = run_scenario(scenario)
    all_results.append(result)

## Summary and save results

In [ ]:
passed_count = sum(1 for r in all_results if r["passed_d4_criterion"])
print(f"\n{'='*60}")
print(f"CAREVOICE -- FINAL RESULTS")
print(f"{'='*60}")
print(f"Scenarios run:    {len(all_results)}")
print(f"Scenarios passed: {passed_count}/{len(all_results)}")
print(f"Pass rate:        {passed_count / len(all_results):.0%}")

red_flag_scenarios = [r for r in all_results if r["final_record"]["red_flags"]]
print(f"Red flags raised: {len(red_flag_scenarios)} scenarios")

avg_confidence = sum(r["avg_confidence"] for r in all_results) / len(all_results)
print(f"Avg confidence:   {avg_confidence:.3f}")

# Language coverage
lang_counts = {}
for r in all_results:
    sid = r["scenario_id"]
    sc = next(s for s in SCENARIOS if s["scenario_id"] == sid)
    lang_counts[sc["language"]] = lang_counts.get(sc["language"], 0) + 1
print(f"\nLanguage coverage: {lang_counts}")

# Accessibility: model fits 8GB RAM at 4-bit (e4b-it = 4B params = ~3GB 4-bit)
print(f"Model:            {'gemma-4-e4b-it' if 'e4b' in str(MODEL_PATH) else 'gemma-4-e2b-it'}")
print(f"Device:           {DEVICE}")
if torch.cuda.is_available() and USE_GPU:
    print(f"GPU memory used:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")

output = {
    "project": "CareVoice",
    "description": "Offline clinical intake assistant -- Gemma 4, multilingual, 8GB RAM",
    "competition": "gemma-4-good-hackathon",
    "scenarios_run": len(all_results),
    "passed_count": passed_count,
    "all_passed": passed_count == len(all_results),
    "avg_confidence": round(avg_confidence, 3),
    "red_flag_scenarios": len(red_flag_scenarios),
    "language_coverage": lang_counts,
    "model": str(MODEL_PATH),
    "device": DEVICE,
    "results": all_results,
}

output_path = Path("/kaggle/working/carevoice_results.json")
output_path.write_text(json.dumps(output, indent=2, default=str), encoding="utf-8")
print(f"\nResults saved to: {output_path}")
print(f"\nCareVoice: {'PASS' if output['all_passed'] else 'FAIL'}")